# Prepare the Summary Inputs

| Item | Description |
|---|---|
| Goal | Fix everything the per-filing summary stage (T3.7.2) and the per-firm stage (T3.7.4) need before any model call: the segments each filing is read in, the prompts of the three summary families for the map, collapse, and reduce stages, the generation settings and budgets, and the input shards. |
| Input | `output/T3.4.1/filings.csv` (the sampled filings) and `sp500_10k_item1.parquet` (Item 1 text) under `/project2/mmiric_1435/acroh/llm_method/data`. |
| Output | `output/T3.7.1/prompts.md` and `settings.json` (readable), `output/T3.7.1/token_counts.csv` (one row per filing with its segment count), and on scratch `<scratch>/T3.7 - Firm Summaries/Interim/filing_inputs/<set>/shard_<k>.parquet` (one row per segment) with `manifest.csv`. |
| Method | The map-reduce of the hearing summaries (T1): a filing's Item 1 is cut into segments of at most 4,000 model tokens at paragraph boundaries, as T1 cuts a hearing into turn-aware chunks; each segment is summarized (map); when the segment summaries together exceed the 6,144-token aggregation budget they are condensed in groups (collapse) until they fit; one call then writes the filing's summary (reduce). The three stages have distinct prompts, as in T1. |
| Families | `summary_without_focus`: what the company does, for 1B. `summary_lines_of_business`: one paragraph on the products, services, or segments the company sells or operates and its customers, for Track 2 (version 1.1 returned a numbered list; 1.2 a paragraph with a customer-data sentence where the text had one, which the model turned into a closing "says nothing about customer data" sentence in 54% of filings; 1.3 drops the customer-data clause, since `summary_privacy` covers data, and tells the reduce and firm steps to ignore segment summaries that report no business and not to refer to the summaries; the first 1.3 run, made before that last change, was discarded). The 1.1 and 1.2 prompts are kept in `output/T3.7.1/prompts_<version>.md` and `settings_<version>.json`. `summary_privacy`: how the business involves consumer personal data and which privacy laws apply, the advisor's "try one out with privacy". The per-filing summary of every family is asked for in 150 to 200 words so it embeds whole within 512 bge tokens. |
| Input sets | `three_years`: the 2,072 filings of fiscal 2010, 2015, 2020, run first. `remaining`: the other 16,707 filings of the all-years sample, run afterwards. No filing is summarized twice. |
| Model | `Qwen/Qwen3.5-4B` from the project's local snapshot; generation as in T1 and T3.5.4: `do_sample=True`, `temperature=0.1`, reasoning traces off; map at most 320 new tokens, collapse 384, reduce 512. |

See `Topic Relevance Rebuild Plan.md` in the TEMP folder for the surrounding steps.


## Step 1. Import packages


In [1]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from transformers import AutoTokenizer
from IPython.display import Markdown, display


## Step 2. Define file locations and settings

The budgets are T1's: 4,000-token segments, a 6,144-token aggregation budget, and the three output caps. Every value is named here and copied into `settings.json`.


In [2]:
PROJECT_FOLDER = Path("/project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis")
SOURCE_DATA_FOLDER = Path("/project2/mmiric_1435/acroh/llm_method/data")
SCRATCH_FOLDER = Path("/scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3-TEMP")
NOTEBOOK_SUBFOLDER = Path("T3 - Topic Relevance and Corporate Disclosure/TEMP")
NOTEBOOK_NAME = "T3.7.1"  # the identifier names this notebook's output folder
FILINGS_FILE = PROJECT_FOLDER / NOTEBOOK_SUBFOLDER / "output/T3.4.1/filings.csv"
ITEM1_FILE = SOURCE_DATA_FOLDER / "interim/repo_data/raw/03_sp500_10k_item1/sp500_10k_item1.parquet"
MODEL_FOLDER = Path(
    "/scratch2/acroh/projects/llm_method/hf_cache/hub/models--Qwen--Qwen3.5-4B"
    "/snapshots/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a"
)
INPUTS_FOLDER = SCRATCH_FOLDER / "T3.7 - Firm Summaries" / "Interim" / "filing_inputs"
OUTPUT_FOLDER = PROJECT_FOLDER / NOTEBOOK_SUBFOLDER / "output" / NOTEBOOK_NAME

PROMPT_VERSION = "item1-mapreduce-1.3"  # bump whenever a prompt string below changes; 1.1 tightened the lines-of-business tasks, 1.2 made them one paragraph, 1.3 dropped their customer-data clause and told the reduce and firm steps to ignore empty segments and not refer to the summaries
FAMILIES = ["summary_without_focus", "summary_lines_of_business", "summary_privacy"]
SEGMENT_TOKENS = 4_000  # model tokens per segment, T1's chunk budget
AGGREGATION_BUDGET = 6_144  # model tokens of segment summaries a reduce call may receive, T1's budget
MAP_MAX_NEW_TOKENS = 320  # T1 caps per stage
COLLAPSE_MAX_NEW_TOKENS = 384
REDUCE_MAX_NEW_TOKENS = 512
MAP_TARGET_WORDS = "about 150"
COLLAPSE_TARGET_WORDS = "about 200"
FILING_TARGET_WORDS = "150 to 200"  # per-filing summary; lands inside 512 bge tokens with headroom
FIRM_TARGET_WORDS = "200 to 250"  # per-firm summary built from the filing summaries
MAX_BGE_TOKENS = 512  # the encoder's input limit every final summary must respect
GENERATION = {"do_sample": True, "temperature": 0.1}  # as in T1
N_SHARDS = 8  # one SLURM array task per shard, per input set
INPUTS_FOLDER.mkdir(parents=True, exist_ok=True)
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
print(f"Inputs: {INPUTS_FOLDER}")
print(f"Outputs: {OUTPUT_FOLDER}")


Inputs: /scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3-TEMP/T3.7 - Firm Summaries/Interim/filing_inputs
Outputs: /project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3 - Topic Relevance and Corporate Disclosure/TEMP/output/T3.7.1


## Step 3. Load the sampled filings and their Item 1 text

Every sampled filing must have its text.


In [3]:
filings = pd.read_csv(FILINGS_FILE, dtype={"gvkey": str, "cik": str})
assert filings["accessionNo"].is_unique
item1 = pd.read_parquet(ITEM1_FILE, columns=["accessionNo", "item1"])
filings = filings.merge(item1, on="accessionNo", how="left", validate="one_to_one")
assert filings["item1"].notna().all()
print(f"Filings: {len(filings):,}; firms: {filings['gvkey'].nunique():,}; three-year filings: {int(filings['in_three_years'].sum()):,}")
display(filings[["accessionNo", "gvkey", "conm", "fyear", "in_three_years"]].head(3))


Filings: 18,779; firms: 1,058; three-year filings: 2,072


,accessionNo,gvkey,conm,fyear,in_three_years
0,0000912057-99-011120,001013,ADC TELECOMMUNICATIONS INC,1999,0
1,0000912057-01-000926,001013,ADC TELECOMMUNICATIONS INC,2000,0
2,0000912057-02-001614,001013,ADC TELECOMMUNICATIONS INC,2001,0


## Step 4. Count model tokens per filing

The model's own tokenizer, as T1 insists, because character counts are off by too much on this text.


In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_FOLDER, local_files_only=True)
token_counts = []
texts = filings["item1"].tolist()
for start in range(0, len(texts), 256):
    encoded = tokenizer(texts[start:start + 256], add_special_tokens=False, return_attention_mask=False)
    token_counts.extend(len(ids) for ids in encoded["input_ids"])
filings["n_model_tokens"] = token_counts
print(f"Model tokens per filing: min {filings['n_model_tokens'].min():,}, median {int(filings['n_model_tokens'].median()):,}, max {filings['n_model_tokens'].max():,}; total {filings['n_model_tokens'].sum() / 1e6:.0f} million")


Model tokens per filing: min 217, median 8,260, max 248,449; total 191 million


## Step 5. Write the prompts

Three stages with distinct prompts, as in T1: the map prompt reads one segment of the filing; the collapse prompt condenses a run of segment summaries and is told not to conclude; the reduce prompt writes the filing's summary from all segment summaries. Each family changes only the task. The per-firm prompts (T3.7.4) combine a firm's yearly summaries the same way.


In [5]:
SYSTEM_PROMPTS = {
    "map": "You are an expert reader of corporate 10-K filings. You report only what the text states.",
    "collapse": "You are an expert reader of corporate 10-K filings. You condense summaries without adding to them.",
    "reduce": "You are an expert reader of corporate 10-K filings. You write from the summaries given and add nothing that is not in them.",
}

MAP_OPENING = """Below is one consecutive segment of the Item 1 (Business) section of the 10-K that {company} filed for fiscal year {fyear}. The segments are read in order.

--- SEGMENT {segment} OF {n_segments} ---
{text}
--- END OF SEGMENT ---
"""

MAP_TASKS = {
    "summary_without_focus": """Summarize this segment in {target_words} words: what the company does, its products and services, its customers and markets, how it operates, and the regulations it says it is subject to, as this segment states them. Keep the company's own terms. If the segment contains none of these, for example only headings or boilerplate, say so in one sentence.

Write only the summary of the segment above.""",
    "summary_lines_of_business": """Summarize in {target_words} words of continuous prose what this segment says the company's business is: the products, services, or segments it sells or operates, and who its customers are. Do not describe internal activities such as labor relations, fuel purchasing, insurance, financing, or regulatory compliance. If the segment describes no business, say so in one sentence.

Write only the summary of the segment above.""",
    "summary_privacy": """Report in {target_words} words what this segment says about consumer personal data: which activities collect, store, use, share, or sell personal data; whose data it is; and which privacy or data protection laws and regulators it names. If the segment says nothing about personal data or privacy, say so in one sentence.

Write only the report for the segment above.""",
}

COLLAPSE_OPENING = """Below are summaries of consecutive segments of the Item 1 (Business) section of the 10-K that {company} filed for fiscal year {fyear}, in the order they occur. They are already summaries, not the filing text.

--- SEGMENT SUMMARIES ---
{text}
--- END ---
"""

COLLAPSE_TASKS = {
    "summary_without_focus": """Condense them into one account of {target_words} words that keeps what the company does, its products and services, customers and markets, operations, and the regulations named. This is a portion of the section, so do not write an overall conclusion about the company.

Write only the condensed account.""",
    "summary_lines_of_business": """Condense them into one paragraph of {target_words} words that keeps every distinct product, service, or segment named, removing repetition. This is a portion of the section, so do not write an overall conclusion about the company.

Write only the condensed paragraph.""",
    "summary_privacy": """Condense them into one account of {target_words} words that keeps every activity involving personal data, whose data it is, and every privacy law and regulator named, removing repetition. This is a portion of the section, so do not write an overall conclusion about the company.

Write only the condensed account.""",
}

REDUCE_OPENING = """Below are summaries of every segment of the Item 1 (Business) section of the 10-K that {company} filed for fiscal year {fyear}, in order. They are already summaries, not the filing text.

--- SEGMENT SUMMARIES ---
{text}
--- END ---
"""

REDUCE_TASKS = {
    "summary_without_focus": """Write a single summary of the whole section in {target_words} words of continuous prose: what the company does, its products and services, its customers and markets, how it operates, and the regulations it says it is subject to. Keep the company's own terms. Where the segment summaries do not establish one of these, leave it out rather than inferring it.

Write only the summary.""",
    "summary_lines_of_business": """Write one paragraph of {target_words} words of continuous prose describing the company's business as a whole: the products, services, or segments it sells or operates, and who its customers are, with no list, no headings, and no repetition. Include only what the summaries state. Ignore any segment summary that says the segment describes no business. Write about the company only; do not refer to the summaries or to what they do not say. Do not add a closing or general statement.

Write only the paragraph.""",
    "summary_privacy": """Write a single account of the whole section in {target_words} words of continuous prose: how the company's business involves consumer personal data, which activities collect, store, use, share, or sell personal data, whose data it is, and which privacy or data protection laws and regulators are named. If the segment summaries say the section does not mention personal data or privacy, write exactly one sentence saying so and nothing else. Where the segment summaries do not establish something, leave it out rather than inferring it.

Write only the account.""",
}

FIRM_OPENING = """Below are summaries of the Item 1 (Business) sections of the 10-Ks that {company} filed for fiscal years {years}, one per year and marked with its year. They are already summaries, not the filing text.

--- YEARLY SUMMARIES ---
{text}
--- END ---
"""

FIRM_TASKS = {
    "summary_without_focus": """Write a single description of the company's business over these years in {target_words} words of continuous prose: what the company does, its products and services, its customers and markets, how it operates, and the regulations it says it is subject to, noting changes across the years where the summaries show them. Add nothing that is not in the summaries.

Write only the description.""",
    "summary_lines_of_business": """Write one paragraph of {target_words} words of continuous prose describing the company's business across these years, noting changes where the summaries show them, with no list and no repetition. Include only what the summaries state. Write about the company only; do not refer to the summaries or to what they do not say. Do not add a closing or general statement.

Write only the paragraph.""",
    "summary_privacy": """Write a single account in {target_words} words of continuous prose of how the company's business involves consumer personal data across these years: which activities collect, store, use, share, or sell personal data, whose data it is, and which privacy or data protection laws and regulators are named. If the summaries say the filings do not mention personal data or privacy, write exactly one sentence saying so and nothing else. Add nothing that is not in the summaries.

Write only the account.""",
}

prompt_markdown = ""
for stage, system_prompt in SYSTEM_PROMPTS.items():
    prompt_markdown += f"**System prompt, {stage}**\n\n{system_prompt}\n\n"
for title, opening, tasks in [("Map (one segment)", MAP_OPENING, MAP_TASKS), ("Collapse (a run of segment summaries)", COLLAPSE_OPENING, COLLAPSE_TASKS),
                              ("Reduce (all segment summaries of a filing)", REDUCE_OPENING, REDUCE_TASKS), ("Per firm (the firm's yearly summaries)", FIRM_OPENING, FIRM_TASKS)]:
    prompt_markdown += f"**{title}, opening**\n\n```\n{opening}```\n\n"
    for family, task in tasks.items():
        prompt_markdown += f"**{title}, task `{family}`**\n\n```\n{task}\n```\n\n"
display(Markdown(prompt_markdown))


**System prompt, map**

You are an expert reader of corporate 10-K filings. You report only what the text states.

**System prompt, collapse**

You are an expert reader of corporate 10-K filings. You condense summaries without adding to them.

**System prompt, reduce**

You are an expert reader of corporate 10-K filings. You write from the summaries given and add nothing that is not in them.

**Map (one segment), opening**

```
Below is one consecutive segment of the Item 1 (Business) section of the 10-K that {company} filed for fiscal year {fyear}. The segments are read in order.

--- SEGMENT {segment} OF {n_segments} ---
{text}
--- END OF SEGMENT ---
```

**Map (one segment), task `summary_without_focus`**

```
Summarize this segment in {target_words} words: what the company does, its products and services, its customers and markets, how it operates, and the regulations it says it is subject to, as this segment states them. Keep the company's own terms. If the segment contains none of these, for example only headings or boilerplate, say so in one sentence.

Write only the summary of the segment above.
```

**Map (one segment), task `summary_lines_of_business`**

```
Summarize in {target_words} words of continuous prose what this segment says the company's business is: the products, services, or segments it sells or operates, and who its customers are. Do not describe internal activities such as labor relations, fuel purchasing, insurance, financing, or regulatory compliance. If the segment describes no business, say so in one sentence.

Write only the summary of the segment above.
```

**Map (one segment), task `summary_privacy`**

```
Report in {target_words} words what this segment says about consumer personal data: which activities collect, store, use, share, or sell personal data; whose data it is; and which privacy or data protection laws and regulators it names. If the segment says nothing about personal data or privacy, say so in one sentence.

Write only the report for the segment above.
```

**Collapse (a run of segment summaries), opening**

```
Below are summaries of consecutive segments of the Item 1 (Business) section of the 10-K that {company} filed for fiscal year {fyear}, in the order they occur. They are already summaries, not the filing text.

--- SEGMENT SUMMARIES ---
{text}
--- END ---
```

**Collapse (a run of segment summaries), task `summary_without_focus`**

```
Condense them into one account of {target_words} words that keeps what the company does, its products and services, customers and markets, operations, and the regulations named. This is a portion of the section, so do not write an overall conclusion about the company.

Write only the condensed account.
```

**Collapse (a run of segment summaries), task `summary_lines_of_business`**

```
Condense them into one paragraph of {target_words} words that keeps every distinct product, service, or segment named, removing repetition. This is a portion of the section, so do not write an overall conclusion about the company.

Write only the condensed paragraph.
```

**Collapse (a run of segment summaries), task `summary_privacy`**

```
Condense them into one account of {target_words} words that keeps every activity involving personal data, whose data it is, and every privacy law and regulator named, removing repetition. This is a portion of the section, so do not write an overall conclusion about the company.

Write only the condensed account.
```

**Reduce (all segment summaries of a filing), opening**

```
Below are summaries of every segment of the Item 1 (Business) section of the 10-K that {company} filed for fiscal year {fyear}, in order. They are already summaries, not the filing text.

--- SEGMENT SUMMARIES ---
{text}
--- END ---
```

**Reduce (all segment summaries of a filing), task `summary_without_focus`**

```
Write a single summary of the whole section in {target_words} words of continuous prose: what the company does, its products and services, its customers and markets, how it operates, and the regulations it says it is subject to. Keep the company's own terms. Where the segment summaries do not establish one of these, leave it out rather than inferring it.

Write only the summary.
```

**Reduce (all segment summaries of a filing), task `summary_lines_of_business`**

```
Write one paragraph of {target_words} words of continuous prose describing the company's business as a whole: the products, services, or segments it sells or operates, and who its customers are, with no list, no headings, and no repetition. Include only what the summaries state. Ignore any segment summary that says the segment describes no business. Write about the company only; do not refer to the summaries or to what they do not say. Do not add a closing or general statement.

Write only the paragraph.
```

**Reduce (all segment summaries of a filing), task `summary_privacy`**

```
Write a single account of the whole section in {target_words} words of continuous prose: how the company's business involves consumer personal data, which activities collect, store, use, share, or sell personal data, whose data it is, and which privacy or data protection laws and regulators are named. If the segment summaries say the section does not mention personal data or privacy, write exactly one sentence saying so and nothing else. Where the segment summaries do not establish something, leave it out rather than inferring it.

Write only the account.
```

**Per firm (the firm's yearly summaries), opening**

```
Below are summaries of the Item 1 (Business) sections of the 10-Ks that {company} filed for fiscal years {years}, one per year and marked with its year. They are already summaries, not the filing text.

--- YEARLY SUMMARIES ---
{text}
--- END ---
```

**Per firm (the firm's yearly summaries), task `summary_without_focus`**

```
Write a single description of the company's business over these years in {target_words} words of continuous prose: what the company does, its products and services, its customers and markets, how it operates, and the regulations it says it is subject to, noting changes across the years where the summaries show them. Add nothing that is not in the summaries.

Write only the description.
```

**Per firm (the firm's yearly summaries), task `summary_lines_of_business`**

```
Write one paragraph of {target_words} words of continuous prose describing the company's business across these years, noting changes where the summaries show them, with no list and no repetition. Include only what the summaries state. Write about the company only; do not refer to the summaries or to what they do not say. Do not add a closing or general statement.

Write only the paragraph.
```

**Per firm (the firm's yearly summaries), task `summary_privacy`**

```
Write a single account in {target_words} words of continuous prose of how the company's business involves consumer personal data across these years: which activities collect, store, use, share, or sell personal data, whose data it is, and which privacy or data protection laws and regulators are named. If the summaries say the filings do not mention personal data or privacy, write exactly one sentence saying so and nothing else. Add nothing that is not in the summaries.

Write only the account.
```



## Step 6. Cut each filing into segments

Paragraphs (blank-line blocks) are packed whole into segments of at most 4,000 model tokens, in order, as T1 packs whole speaking turns. A paragraph longer than a segment is cut into token windows, each re-encoded to confirm it fits, because a decoded slice can re-encode longer than it was cut. Every segment is re-encoded before it is written.


In [6]:
JOIN_TEXT = "\n\n"
join_cost = len(tokenizer(JOIN_TEXT, add_special_tokens=False)["input_ids"])


def encoded_length(text):
    return len(tokenizer(text, add_special_tokens=False)["input_ids"])


def token_windows(text, max_tokens):
    """Cut text into pieces that each re-encode within max_tokens."""
    ids = tokenizer(text, add_special_tokens=False)["input_ids"]
    pieces, start = [], 0
    while start < len(ids):
        take = min(max_tokens, len(ids) - start)
        piece = tokenizer.decode(ids[start:start + take])
        while take > 64 and encoded_length(piece) > max_tokens:
            take = int(take * 0.9)
            piece = tokenizer.decode(ids[start:start + take])
        pieces.append(piece)
        start += take
    return pieces


def segment_item1(text, max_tokens):
    """Pack whole paragraphs into segments of at most max_tokens model tokens."""
    paragraphs = [block.strip() for block in re.split(r"\n\s*\n|\r\n\s*\r\n", text) if block.strip()]
    lengths = [len(ids) for ids in tokenizer(paragraphs, add_special_tokens=False, return_attention_mask=False)["input_ids"]]
    segments, current, current_tokens = [], [], 0
    for paragraph, n_tokens in zip(paragraphs, lengths):
        if n_tokens > max_tokens:
            if current:
                segments.append(JOIN_TEXT.join(current))
                current, current_tokens = [], 0
            segments.extend(token_windows(paragraph, max_tokens))
            continue
        cost = n_tokens + (join_cost if current else 0)
        if current_tokens + cost > max_tokens:
            segments.append(JOIN_TEXT.join(current))
            current, current_tokens = [], 0
            cost = n_tokens
        current.append(paragraph)
        current_tokens += cost
    if current:
        segments.append(JOIN_TEXT.join(current))
    verified = []
    for segment in segments:
        verified.extend([segment] if encoded_length(segment) <= max_tokens else token_windows(segment, max_tokens))
    return verified


segment_rows = []
for filing in filings.itertuples(index=False):
    segments = segment_item1(filing.item1, SEGMENT_TOKENS)
    for segment_id, segment_text in enumerate(segments, start=1):
        segment_rows.append({
            "accessionNo": filing.accessionNo, "gvkey": filing.gvkey, "conm": filing.conm, "fyear": filing.fyear,
            "in_three_years": filing.in_three_years, "segment_id": segment_id, "n_segments": len(segments), "text": segment_text,
        })
segments_table = pd.DataFrame(segment_rows)
segments_table["n_model_tokens"] = [
    len(ids) for start in range(0, len(segments_table), 256)
    for ids in tokenizer(segments_table["text"].iloc[start:start + 256].tolist(), add_special_tokens=False, return_attention_mask=False)["input_ids"]
]
assert segments_table["n_model_tokens"].le(SEGMENT_TOKENS).all()
assert segments_table.groupby("accessionNo")["segment_id"].count().eq(segments_table.groupby("accessionNo")["n_segments"].first()).all()
segments_per_filing = segments_table.groupby("accessionNo")["n_segments"].first()
print(f"Segments: {len(segments_table):,} for {segments_per_filing.size:,} filings; per filing min {segments_per_filing.min()}, median {int(segments_per_filing.median())}, max {segments_per_filing.max()}; largest segment {segments_table['n_model_tokens'].max():,} tokens")


Segments: 57,754 for 18,779 filings; per filing min 1, median 3, max 69; largest segment 4,000 tokens


## Step 7. Write the input sets and their shards

The three-year filings form the first set; the rest of the all-years sample forms the second. Each set is cut into eight shards of filings; a filing's segments stay in one shard.


In [7]:
manifest_rows = []
for set_name, mask in [("three_years", segments_table["in_three_years"].eq(1)), ("remaining", segments_table["in_three_years"].eq(0))]:
    set_segments = segments_table.loc[mask].copy()
    set_filings = set_segments["accessionNo"].drop_duplicates().reset_index(drop=True)
    shard_of_filing = pd.Series((np.arange(len(set_filings)) * N_SHARDS // len(set_filings)).astype(int), index=set_filings)
    set_segments["shard_id"] = set_segments["accessionNo"].map(shard_of_filing)
    set_folder = INPUTS_FOLDER / set_name
    set_folder.mkdir(parents=True, exist_ok=True)
    for shard_id, shard in set_segments.groupby("shard_id", sort=True):
        shard_file = set_folder / f"shard_{shard_id}.parquet"
        shard[["accessionNo", "gvkey", "conm", "fyear", "segment_id", "n_segments", "n_model_tokens", "text"]].to_parquet(shard_file, index=False)
        manifest_rows.append({
            "set": set_name, "shard_id": int(shard_id), "n_filings": int(shard["accessionNo"].nunique()),
            "n_segments": int(len(shard)), "n_model_tokens": int(shard["n_model_tokens"].sum()), "file": f"{set_name}/{shard_file.name}",
        })
manifest = pd.DataFrame(manifest_rows)
assert manifest["n_segments"].sum() == len(segments_table)
manifest.to_csv(INPUTS_FOLDER / "manifest.csv", index=False)
manifest.to_csv(OUTPUT_FOLDER / "manifest.csv", index=False)
print(f"Saved: {INPUTS_FOLDER / 'manifest.csv'}")
display(manifest)


Saved: /scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3-TEMP/T3.7 - Firm Summaries/Interim/filing_inputs/manifest.csv


,set,shard_id,n_filings,n_segments,n_model_tokens,file
0,three_years,0,259,716,2345757,three_years/shard_0.parquet
1,three_years,1,259,788,2592632,three_years/shard_1.parquet
2,three_years,2,259,692,2243470,three_years/shard_2.parquet
3,three_years,3,259,768,2516616,three_years/shard_3.parquet
4,three_years,4,259,947,3207326,three_years/shard_4.parquet
5,three_years,5,259,840,2830795,three_years/shard_5.parquet
6,three_years,6,259,845,2847896,three_years/shard_6.parquet
7,three_years,7,259,884,2991529,three_years/shard_7.parquet
8,remaining,0,2089,5743,18559444,remaining/shard_0.parquet
9,remaining,1,2088,5901,19012745,remaining/shard_1.parquet


## Step 8. Save the prompts, settings, and token counts

The Markdown file is for reading; `settings.json` is what the stage scripts read; `token_counts.csv` records every filing's size and segment count.


In [8]:
settings = {
    "prompt_version": PROMPT_VERSION,
    "model_folder": str(MODEL_FOLDER),
    "families": FAMILIES,
    "generation": GENERATION,
    "enable_thinking": False,
    "segment_tokens": SEGMENT_TOKENS,
    "aggregation_budget": AGGREGATION_BUDGET,
    "max_new_tokens": {"map": MAP_MAX_NEW_TOKENS, "collapse": COLLAPSE_MAX_NEW_TOKENS, "reduce": REDUCE_MAX_NEW_TOKENS},
    "target_words": {"map": MAP_TARGET_WORDS, "collapse": COLLAPSE_TARGET_WORDS, "reduce": FILING_TARGET_WORDS, "firm": FIRM_TARGET_WORDS},
    "max_bge_tokens": MAX_BGE_TOKENS,
    "prompts": {
        "system": SYSTEM_PROMPTS,
        "map_opening": MAP_OPENING, "map_tasks": MAP_TASKS,
        "collapse_opening": COLLAPSE_OPENING, "collapse_tasks": COLLAPSE_TASKS,
        "reduce_opening": REDUCE_OPENING, "reduce_tasks": REDUCE_TASKS,
        "firm_opening": FIRM_OPENING, "firm_tasks": FIRM_TASKS,
    },
    "inputs_folder": str(INPUTS_FOLDER),
}
(OUTPUT_FOLDER / "settings.json").write_text(json.dumps(settings, indent=2, ensure_ascii=False), encoding="utf-8")
(OUTPUT_FOLDER / "prompts.md").write_text(
    f"# Firm summary prompts and settings ({PROMPT_VERSION})\n\n"
    f"{len(filings):,} filings cut into {len(segments_table):,} segments of at most {SEGMENT_TOKENS:,} model tokens; "
    f"aggregation budget {AGGREGATION_BUDGET:,} tokens; per-filing summaries of {FILING_TARGET_WORDS} words, per-firm summaries of {FIRM_TARGET_WORDS} words.\n\n"
    "## Settings\n\n```json\n" + json.dumps({key: value for key, value in settings.items() if key != "prompts"}, indent=2) + "\n```\n\n"
    + prompt_markdown.replace("**", "## ") + "\n",
    encoding="utf-8",
)
filings[["accessionNo", "gvkey", "fyear", "in_three_years", "n_model_tokens"]].merge(
    segments_per_filing.rename("n_segments"), on="accessionNo"
).to_csv(OUTPUT_FOLDER / "token_counts.csv", index=False)
print(f"Saved: {OUTPUT_FOLDER / 'settings.json'}")
print(f"Saved: {OUTPUT_FOLDER / 'prompts.md'}")
print(f"Saved: {OUTPUT_FOLDER / 'token_counts.csv'}")


Saved: /project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3 - Topic Relevance and Corporate Disclosure/TEMP/output/T3.7.1/settings.json
Saved: /project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3 - Topic Relevance and Corporate Disclosure/TEMP/output/T3.7.1/prompts.md
Saved: /project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3 - Topic Relevance and Corporate Disclosure/TEMP/output/T3.7.1/token_counts.csv
